# Build Feature Tables

Combines all modalities into a single wide participant-level feature table for both training and challenge.

**Output:**
- `merged_data/train_features.csv` — one row per training participant, all feature columns
- `merged_data/train_targets.csv` — HAI targets at day 28 and day 365 for training participants
- `merged_data/challenge_features.csv` — one row per challenge participant, same feature columns

**Feature groups:**
1. Demographics: `age`, `biological_sex`, `race`, `geolocation`, `arm_name`
2. HAI baseline (t=0): log2-transformed titer per virus strain
3. Flow cytometry (t=0 and t=7): percentage of each cell population (overlapping names only)
4. Transcriptomics gene signatures (t=0 and t=7): mean TPM of each gene signature

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path('.')
CLEANED = ROOT / 'cleaned_data'
MERGED  = ROOT / 'merged_data'
SIGS    = ROOT / 'data' / 'additional_files'

os.makedirs(MERGED, exist_ok=True)

## 1. Gene Signatures

Parse the `.grp` files. Each file has a comment header line then one Ensembl gene ID per line in the format `ENSG00000004468.CD38.protein_coding`. We strip to the base ID.

In [ ]:
def parse_grp(path):
    """Return list of base Ensembl IDs from a .grp file (skips comment lines)."""
    genes = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            # Format: ENSG00000004468.CD38.protein_coding  or  ENSG00000004468
            genes.append(line.split('.')[0])
    return genes

sig_files = {
    'CHI4':          SIGS / 'CHI4.grp',
    'CHI5':          SIGS / 'CHI5-clean.grp',
    'M156_1':        SIGS / 'M156_1_clean.grp',
    'antibody_sig':  SIGS / 'antibody_signature.grp',
    'POU2AF1':       SIGS / 'POU2AF1.grp',
    'TNFRSF17':      SIGS / 'TNFRSF17.grp',
    'platelet':      SIGS / 'platelet_activation_actin_binding.grp',
}

signatures = {name: parse_grp(path) for name, path in sig_files.items()}

for name, genes in signatures.items():
    print(f'{name}: {len(genes)} genes')

## 2. Transcriptomics Signature Scores

For each training study and for the challenge dataset, compute the mean TPM of each signature's genes at t=0 and t=7. Studies with different gene sets will still work — we average whichever signature genes are present.

In [ ]:
def compute_sig_scores(df_wide, signatures, timepoints):
    """
    Given a wide transcriptomics dataframe (participant_id, timepoint, genes...),
    compute mean TPM per signature per timepoint.
    Returns a participant-level dataframe (one row per participant).
    """
    rows = []
    for _, row in df_wide.iterrows():
        pid = row['participant_id']
        tp  = row['timepoint']
        if tp not in timepoints:
            continue
        entry = {'participant_id': pid}
        for sig_name, genes in signatures.items():
            present = [g for g in genes if g in df_wide.columns]
            vals = row[present].dropna()
            entry[f'trans_t{int(tp)}_{sig_name}'] = vals.mean() if len(vals) > 0 else np.nan
        rows.append(entry)

    df_scores = pd.DataFrame(rows)
    # Pivot: one row per participant with t0 and t7 columns side by side
    df_t = df_scores.set_index('participant_id')
    return df_t

In [ ]:
# Training transcriptomics: process each study, then stack
train_trans_files = {
    '2019_UGA':  CLEANED / 'transcriptomics_2019_UGA_cleaned.csv',
    '2020_UGA':  CLEANED / 'transcriptomics_2020_UGA_cleaned.csv',
    'SDY2867':   CLEANED / 'transcriptomics_SDY2867_cleaned.csv',
    'SDY2941':   CLEANED / 'transcriptomics_SDY2941_cleaned.csv',
}

train_sig_parts = []
for study, path in train_trans_files.items():
    print(f'Processing {study}...')
    df_wide = pd.read_csv(path)
    scores = compute_sig_scores(df_wide, signatures, timepoints=[0, 7])
    train_sig_parts.append(scores)
    print(f'  → {scores.shape[0]} (participant, timepoint) rows')

# Stack all studies; each participant appears only in one study
train_sig_scores = pd.concat(train_sig_parts)
# Aggregate: for each participant, combine t0 and t7 score columns into one row
train_sig_scores = train_sig_scores.groupby('participant_id').first().reset_index()
print('\nTraining sig scores shape:', train_sig_scores.shape)
train_sig_scores.head()

In [ ]:
# Challenge transcriptomics
print('Processing challenge transcriptomics...')
ch_wide = pd.read_csv(CLEANED / 'challenge_transcriptomics_cleaned.csv')
challenge_sig_scores = compute_sig_scores(ch_wide, signatures, timepoints=[0, 7])
challenge_sig_scores = challenge_sig_scores.groupby('participant_id').first().reset_index()
print('Challenge sig scores shape:', challenge_sig_scores.shape)
challenge_sig_scores.head()

## 3. HAI Baseline Features (t=0)

Pivot pre-vaccination HAI titers wide. Apply log2 transform — HAI titers are doubling dilutions (5, 10, 20, ...) so the log2 scale is natural.

In [ ]:
def build_hai_features(df_hai, timepoint=0):
    """Pivot HAI to wide format with log2-transformed values."""
    df_base = df_hai[df_hai['timepoint'] == timepoint].copy()
    df_base['log2_value'] = np.log2(df_base['value'].clip(lower=1))
    df_pivot = df_base.pivot_table(
        index='participant_id',
        columns='virus_strain',
        values='log2_value',
        aggfunc='mean'
    )
    df_pivot.columns = [f'hai_t{timepoint}_{c.replace(" ", "_").replace("/", "-")}'
                        for c in df_pivot.columns]
    df_pivot.columns.name = None
    return df_pivot.reset_index()

In [ ]:
# Training HAI features
train_hai = pd.read_csv(CLEANED / 'hai_cleaned.csv')
train_hai_features = build_hai_features(train_hai, timepoint=0)
print('Training HAI features shape:', train_hai_features.shape)
train_hai_features.head()

In [ ]:
# Challenge HAI features
ch_hai = pd.read_csv(CLEANED / 'challenge_hai_cleaned.csv')
challenge_hai_features = build_hai_features(ch_hai, timepoint=0)
print('Challenge HAI features shape:', challenge_hai_features.shape)
challenge_hai_features.head()

## 4. Flow Cytometry Features (t=0 and t=7)

Use only `percentage` unit (the only unit available in challenge data). Use only cell population names that appear in **both** training and challenge to avoid NaN-only feature columns during training.

Overlapping names (17 total): `Ab-secreting cells ASC`, `B cells`, `CD4 T cells`, `CD4 Tem`, `CD8 T cells`, `CD8 Tcm`, `CD8 Tem`, `Classical monocytes`, `Conventional DC`, `Memory class switched B cells`, `Memory nonclass switched B cells`, `Naive B cells`, `Neutrophils`, `Plasmacytoid DCs`, `T cells`, `Tfh CXCR5+ CD4 T cell`, `Th1 CXCR3+ CD4 T cells`

In [ ]:
FLOW_OVERLAP_NAMES = [
    'Ab-secreting cells ASC',
    'B cells',
    'CD4 T cells',
    'CD4 Tem',
    'CD8 T cells',
    'CD8 Tcm',
    'CD8 Tem',
    'Classical monocytes',
    'Conventional DC',
    'Memory class switched B cells',
    'Memory nonclass switched B cells',
    'Naive B cells',
    'Neutrophils',
    'Plasmacytoid DCs',
    'T cells',
    'Tfh CXCR5+ CD4 T cell',
    'Th1 CXCR3+ CD4 T cells',
]

def build_flow_features(df_flow, timepoints=[0, 7]):
    """Pivot flow cytometry to wide format (percentage unit, overlapping names only)."""
    df_pct = df_flow[
        (df_flow['unit'] == 'percentage') &
        (df_flow['name'].isin(FLOW_OVERLAP_NAMES)) &
        (df_flow['timepoint'].isin(timepoints))
    ].copy()

    df_pivot = df_pct.pivot_table(
        index='participant_id',
        columns=['timepoint', 'name'],
        values='value',
        aggfunc='mean'
    )
    # Flatten multi-level columns
    df_pivot.columns = [
        f'flow_t{int(tp)}_{name.replace(" ", "_")}'
        for tp, name in df_pivot.columns
    ]
    df_pivot.columns.name = None
    return df_pivot.reset_index()

In [ ]:
# Training flow features
train_flow = pd.read_csv(CLEANED / 'flow_cleaned.csv', low_memory=False)
train_flow_features = build_flow_features(train_flow)
print('Training flow features shape:', train_flow_features.shape)
train_flow_features.head()

In [ ]:
# Challenge flow features
ch_flow = pd.read_csv(CLEANED / 'challenge_flow_cleaned.csv', low_memory=False)
challenge_flow_features = build_flow_features(ch_flow)
print('Challenge flow features shape:', challenge_flow_features.shape)
challenge_flow_features.head()

## 5. Assemble Feature Tables

In [ ]:
def assemble_features(participants, hai_features, flow_features, sig_scores):
    """Left-join all feature tables onto participants."""
    df = participants.copy()
    df = df.merge(hai_features,  on='participant_id', how='left')
    df = df.merge(flow_features, on='participant_id', how='left')
    df = df.merge(sig_scores,    on='participant_id', how='left')
    return df

In [ ]:
# Training features
train_part = pd.read_csv(CLEANED / 'participants_cleaned.csv')
train_features = assemble_features(
    train_part,
    train_hai_features,
    train_flow_features,
    train_sig_scores,
)
print('Training feature table shape:', train_features.shape)
train_features.head()

In [ ]:
# Challenge features
ch_part = pd.read_csv(CLEANED / 'challenge_participants_cleaned.csv')
challenge_features = assemble_features(
    ch_part,
    challenge_hai_features,
    challenge_flow_features,
    challenge_sig_scores,
)
print('Challenge feature table shape:', challenge_features.shape)
challenge_features.head()

## 6. Build Training Targets

Extract HAI values at day 28 and day 365 for the three vaccine strains listed in tasks.md. Also compute log2 fold-change (day28 / day0) and geometric mean across vaccine strains.

In [ ]:
VACCINE_STRAINS = [
    'H1N1 A/Victoria/4897/2022',
    'H3N2 A/Massachusetts/18/2022',
    'Vic B/Austria/1359417/2021',
]

def build_hai_targets(df_hai, timepoints=[28, 365]):
    """Pivot post-vaccination HAI (log2) and compute derived targets."""
    df_post = df_hai[df_hai['timepoint'].isin(timepoints)].copy()
    df_post['log2_value'] = np.log2(df_post['value'].clip(lower=1))

    df_pivot = df_post.pivot_table(
        index='participant_id',
        columns=['timepoint', 'virus_strain'],
        values='log2_value',
        aggfunc='mean'
    )
    df_pivot.columns = [
        f'hai_t{int(tp)}_{strain.replace(" ", "_").replace("/", "-")}'
        for tp, strain in df_pivot.columns
    ]
    df_pivot.columns.name = None
    df_targets = df_pivot.reset_index()

    # Also add baseline (t=0) for fold-change calculation
    df_base = df_hai[df_hai['timepoint'] == 0].copy()
    df_base['log2_value'] = np.log2(df_base['value'].clip(lower=1))
    df_base_pivot = df_base.pivot_table(
        index='participant_id',
        columns='virus_strain',
        values='log2_value',
        aggfunc='mean'
    )
    df_base_pivot.columns = [
        f'hai_t0_{c.replace(" ", "_").replace("/", "-")}'
        for c in df_base_pivot.columns
    ]
    df_base_pivot = df_base_pivot.reset_index()
    df_targets = df_targets.merge(df_base_pivot, on='participant_id', how='left')

    # Compute log2 fold-change (t28 / t0) for vaccine strains present in training
    for strain in VACCINE_STRAINS:
        col_safe = strain.replace(' ', '_').replace('/', '-')
        t0_col  = f'hai_t0_{col_safe}'
        t28_col = f'hai_t28_{col_safe}'
        if t0_col in df_targets.columns and t28_col in df_targets.columns:
            df_targets[f'log2fc_t28_{col_safe}'] = df_targets[t28_col] - df_targets[t0_col]

    # Task 4.4: geo mean of log2 HAI at t28 across vaccine strains (= mean of log2 values)
    t28_vaccine_cols = [
        f'hai_t28_{s.replace(" ", "_").replace("/", "-")}'
        for s in VACCINE_STRAINS
        if f'hai_t28_{s.replace(" ", "_").replace("/", "-")}' in df_targets.columns
    ]
    if t28_vaccine_cols:
        df_targets['task4_4_geomean_log2_t28'] = df_targets[t28_vaccine_cols].mean(axis=1)

    # Task 4.5: geo mean log2 HAI across ALL strains at t28
    all_t28_cols = [c for c in df_targets.columns if c.startswith('hai_t28_')]
    if all_t28_cols:
        df_targets['task4_5_geomean_log2_t28_all'] = df_targets[all_t28_cols].mean(axis=1)

    return df_targets

In [ ]:
train_targets = build_hai_targets(train_hai)
print('Training targets shape:', train_targets.shape)
print('Columns:', list(train_targets.columns[:10]), '...')
train_targets.head()

## 7. Save Outputs

In [ ]:
train_features.to_csv(MERGED / 'train_features.csv', index=False)
train_targets.to_csv(MERGED / 'train_targets.csv', index=False)
challenge_features.to_csv(MERGED / 'challenge_features.csv', index=False)

print('Saved:')
print(f'  train_features.csv    {train_features.shape}')
print(f'  train_targets.csv     {train_targets.shape}')
print(f'  challenge_features.csv {challenge_features.shape}')

## 8. Feature Coverage Summary

How many participants have non-null values for each feature group?

In [ ]:
def coverage_summary(df, label):
    demo_cols  = ['age', 'biological_sex', 'race', 'geolocation', 'arm_name']
    hai_cols   = [c for c in df.columns if c.startswith('hai_t0_')]
    flow_cols  = [c for c in df.columns if c.startswith('flow_')]
    trans_cols = [c for c in df.columns if c.startswith('trans_')]

    groups = {
        'Demographics':   demo_cols,
        'HAI t=0':        hai_cols,
        'Flow':           flow_cols,
        'Transcriptomics': trans_cols,
    }
    print(f'\n=== {label} (n={len(df)}) ===')
    for group, cols in groups.items():
        present = [c for c in cols if c in df.columns]
        if not present:
            print(f'  {group}: no columns')
            continue
        non_null = df[present].notna().any(axis=1).sum()
        print(f'  {group}: {non_null}/{len(df)} participants have >=1 non-null value ({len(present)} cols)')

coverage_summary(train_features, 'Training')
coverage_summary(challenge_features, 'Challenge')